In [ ]:
# RQ2: Model Comparison
# Which supervised learning model achieves the best predictive performance?

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/kaggle/input/marketing-and-product-performance-dataset/marketing_and_product_performance.csv')
for col in ['Subscription_Tier', 'Common_Keywords']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df = df.drop(columns=['Campaign_ID', 'Product_ID', 'Customer_ID', 'Flash_Sale_ID', 'Bundle_ID'])
X = df.drop(columns=['Units_Sold'])
y = df['Units_Sold']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'K-NN': KNeighborsRegressor(n_neighbors=5),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost (GB)': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR': SVR(kernel='rbf', C=1.0)
}

results = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    results.append({
        'Model': name,
        'MAE': round(mean_absolute_error(y_test, preds),4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)),4),
        'R2': round(r2_score(y_test, preds),4)
    })

res_df = pd.DataFrame(results).sort_values('R2', ascending=False)
print(res_df)
res_df.to_csv('RQ2_model_comparison.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
colors = plt.cm.Set2(np.linspace(0, 1, len(res_df)))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'R2']):
    vals = res_df[metric]
    bars = ax.barh(res_df['Model'], vals, color=colors)
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} by Model')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8)
plt.suptitle('RQ2: All-Model Comparison Across Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('RQ2_model_comparison.pdf', dpi=150, bbox_inches='tight')
plt.show()